<a href="https://colab.research.google.com/github/shrivastavay882-glitch/ecom-sales-data-pipeline/blob/main/Realistic_sales_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/retail_sales_v2_dirty.csv')
df.info()                      # Date needs to be converted into datetime
df['Date'] = pd.to_datetime(df['Date'])
df.isnull().sum()              # null values are present
df.head()
df.duplicated().sum()          # duplicates are present
df[df.duplicated()]            # shows the duplicated rows
df = df.dropna()               # null values dropped
df = df.drop_duplicates()
df.info()


## Standardizing casing and removing hidden spaces
# Brand column
df['Brand'].unique()    # checked for unique values and found same brand name appearing more than once
df['Brand'] = df['Brand'].str.upper()  # converted to upper case
df['Brand'] = df['Brand'].str.strip()  # trimmed spaces
df['Brand'] = df['Brand'].replace({'ADDIDAS': 'ADIDAS',
                                   'APPEL':'APPLE',
                                   'PUMMA':'PUMA',
                                   'RELAME': 'REALME' ,
                                   'SAMSNG' : 'SAMSUNG',
                                  'MAYBELINE': 'MAYBELLINE'}
                                  )
sorted(df['Brand'].unique())

# Product_Category
sorted(df['Product_Category'].unique()) # shows multiple identical columns
df['Product_Category'] = df['Product_Category'].str.upper()
df['Product_Category'] = df['Product_Category'].str.strip()
df['Product_Category'] = df['Product_Category'].replace({'HOME & KITCHEN':'HOME AND KITCHEN'})
sorted(df['Product_Category'].unique())

# City
sorted(df['City'].unique())
df['City'] = df['City'].str.upper()
df['City'] = df['City'].str.strip()
df['City'] = df['City'].replace({'BENGALURU':'BANGALORE',
                                 'MYSURU':'MYSORE'})
sorted(df['City'].unique())

# product_name
sorted(df['Product_Name'].unique())  # seems good

df.describe()  # Found an unusual quantity as 500

df_filtered = df[df['Quantity']>400]  # Checking that particular row
print(df_filtered)         # Its a bulk order of books so seems ok

df.groupby('Brand')['Quantity'].sum().sort_values(ascending = False)
df.groupby('Product_Category')['Price'].mean().sort_values(ascending = False)

df.to_csv('cleaned_retail_sales_master.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Order_ID          20000 non-null  object
 1   Date              19992 non-null  object
 2   Customer_ID       19989 non-null  object
 3   City              20000 non-null  object
 4   Product_Category  20000 non-null  object
 5   Brand             20000 non-null  object
 6   Product_Name      20000 non-null  object
 7   Quantity          20000 non-null  int64 
 8   MRP               20000 non-null  int64 
 9   Discount_Pct      20000 non-null  int64 
 10  Price             20000 non-null  int64 
 11  Total_Amount      20000 non-null  int64 
 12  Payment_Method    19989 non-null  object
 13  Return_Flag       20000 non-null  object
dtypes: int64(5), object(9)
memory usage: 2.1+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 19965 entries, 0 to 19999
Data columns (total 14 columns):
 

In [3]:
from IPython.display import display
import pandas as pd
import sqlite3


df = pd.read_csv('/content/cleaned_retail_sales_master.csv')

# Establishing the cloud-based in-memory SQL database environment
conn = sqlite3.connect(':memory:')


df.to_sql('retail_sales', conn, index=False, if_exists='replace')


#  Brand Wise Revenue Leaderboard

query_brands = """
SELECT
    Brand,
    SUM(Total_Amount) AS Total_Revenue
FROM retail_sales
GROUP BY Brand
ORDER BY Total_Revenue DESC
LIMIT 5;
"""
print("--- TOP 5 BRANDS BY REVENUE ---")
display(pd.read_sql_query(query_brands, conn))



#  Operational Return-Rate Risk Analysis (FIXED CASE BUG)

query_returns = """
SELECT
    Product_Category,
    COUNT(CASE WHEN UPPER(Return_Flag) = 'YES' THEN 1 END) AS Total_Returns,
    COUNT(Order_ID) AS Total_Orders,
    ROUND((COUNT(CASE WHEN UPPER(Return_Flag) = 'YES' THEN 1 END) * 100.0 / COUNT(Order_ID)), 2) AS Return_Rate_Percentage
FROM retail_sales
GROUP BY Product_Category
ORDER BY Return_Rate_Percentage DESC;
"""
print("--- CATEGORY RETURN RATE ANALYSIS ---")
display(pd.read_sql_query(query_returns, conn))


#  Payment Method & Ticket-Size Optimization
query_payments = """
SELECT
    Payment_Method,
    COUNT(Order_ID) AS Total_Transactions,
    SUM(Total_Amount) AS Total_Revenue_Generated,
    ROUND(AVG(Total_Amount), 2) AS Average_Order_Value
FROM retail_sales
GROUP BY Payment_Method
ORDER BY Total_Revenue_Generated DESC;
"""
print("--- PAYMENT CHANNEL PERFORMANCE ---")
display(pd.read_sql_query(query_payments, conn))



--- TOP 5 BRANDS BY REVENUE ---


,Brand,Total_Revenue
0,APPLE,25265371
1,LENOVO,19213826
2,MI,7737363
3,SAMSUNG,6885287
4,LG,5874624


--- CATEGORY RETURN RATE ANALYSIS ---


,Product_Category,Total_Returns,Total_Orders,Return_Rate_Percentage
0,CLOTHING,922,4268,21.60
1,FOOTWEAR,340,2177,15.62
2,HOME AND KITCHEN,303,3129,9.68
3,ELECTRONICS,250,3245,7.70
4,TOYS,105,1383,7.59
5,BEAUTY,107,1961,5.46
6,MOBILES,100,2243,4.46
7,BEAUTY & PERSONAL CARE,10,239,4.18
8,BOOKS,24,1320,1.82


--- PAYMENT CHANNEL PERFORMANCE ---


,Payment_Method,Total_Transactions,Total_Revenue_Generated,Average_Order_Value
0,UPI,7527,49647580,6595.93
1,Credit Card,2627,20754272,7900.37
2,Debit Card,2273,15319728,6739.87
3,Cash on Delivery,4656,13578082,2916.25
4,EMI,1047,11314988,10807.06
5,Net Banking,807,5472096,6780.79
6,Flipkart Pay Later,718,4674077,6509.86
7,Gift Card,310,2327902,7509.36
